In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.read.format('parquet')\
    .load('abfss://bronze@datalakecommerce.dfs.core.windows.net/orders')

### **Deleting _rescued_data column**

In [0]:
df=df.drop("_rescued_data")

### **File's Schema**

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



### **Changing Type of order_date column**

In [0]:
df=df.withColumn('order_date',col("order_date").cast('timestamp'))

### **Adding year column**

In [0]:
df=df.withColumn('year',year(col('order_date')))

### **Data writing**

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .save('abfss://silver@datalakecommerce.dfs.core.windows.net/orders')

In [0]:
%sql
create table if not exists ecommerce.silver.orders
using delta
location 'abfss://silver@datalakecommerce.dfs.core.windows.net/orders'